In [1]:
import os
import re
import gc
import json
import math
import random
import string
import warnings
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
try:
    from transformers import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# =========================================================
# 0. HYPERPARAMETERS & CONFIG
# =========================================================
SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.15

MAX_LEN = 256
EPOCHS = 6
LR_BASE = 2e-5
LR_HEAD = 5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
PATIENCE = 2
LABEL_SMOOTHING = 0.05

MAX_AI_RATIO_TO_HUMAN = 0.25
CHUNKS_PER_HUMAN_TEXT = 2
SHOW_TQDM = True

MODEL_NAME = "microsoft/mdeberta-v3-base"
DISPLAY_NAME = "Proposed mDeBERTa-v3 + Stylometry"
EXPORT_DIR = Path("model_artifacts")
DATASET_FILENAME = "merged_dataset.csv"

CANDIDATE_DATASET_PATHS = [
    "/kaggle/input/datasets/xandertrevor/stylometry/merged_dataset.csv",
    "/kaggle/input/stylometry/merged_dataset.csv",
    "/kaggle/input/merged-dataset/merged_dataset.csv",
    "/kaggle/input/merged_dataset.csv",
    "merged_dataset.csv",
    "research/merged_dataset.csv",
    "experiment/merged_dataset.csv",
    "../experiment/merged_dataset.csv",
    "./merged_dataset.csv",
    "../merged_dataset.csv",
    "backend/data/processed/merged_dataset.csv",
    "data/processed/merged_dataset.csv",
]

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =========================================================
# 1. LOAD AND PREPARE DATA
# =========================================================
def find_dataset():
    for p in CANDIDATE_DATASET_PATHS:
        if Path(p).exists():
            return p
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob(DATASET_FILENAME))
        if matches:
            return str(matches[0])
    return None

dataset_path = find_dataset()
if not dataset_path:
    raise RuntimeError("Could not find 'merged_dataset.csv'.")

print(f"Loading merged dataset from: {dataset_path}")
df_master = pd.read_csv(dataset_path)

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_master["text"] = df_master["text"].apply(clean_text)

df_human = df_master[df_master["author"] != "AI"].copy()
df_ai = df_master[df_master["author"] == "AI"].copy()

# Dynamic Top-10 Human Authors Filtering
top_human_counts = df_human["author"].value_counts().head(10)
top_human_authors = top_human_counts.index.tolist()
print(f"Selected top {len(top_human_authors)} human authors based on article count:")
for idx, (author, count) in enumerate(top_human_counts.items(), 1):
    print(f"  {idx:02d}. {author} ({count} articles)")

df_human_filtered = df_human[df_human["author"].isin(top_human_authors)].copy()

# Strictly 25% AI ratio
max_ai_rows = int(len(df_human_filtered) * MAX_AI_RATIO_TO_HUMAN)
if len(df_ai) > 0 and max_ai_rows > 0:
    df_ai_sampled = df_ai.sample(n=min(len(df_ai), max_ai_rows), random_state=SEED).copy()
else:
    df_ai_sampled = df_ai.iloc[0:0].copy()

final_df = pd.concat([df_human_filtered, df_ai_sampled], ignore_index=True)
final_df = final_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

labels = sorted(final_df["author"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
final_df["label"] = final_df["author"].map(label2id).astype(int)
NUM_CLASSES = len(labels)

print(f"Unified dataset contains {len(final_df)} rows across {NUM_CLASSES} classes (AI + {NUM_CLASSES - 1} humans).")

# =========================================================
# 2. STYLOMETRIC FEATURE EXTRACTION
# =========================================================
WORD_RE = re.compile(r"[\w]+", re.UNICODE)
SENTENCE_RE = re.compile(r"[^.!?]+[.!?]*", re.UNICODE)
FUNCTION_WORDS = [
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "pada",
    "ini", "itu", "tidak", "akan", "juga", "karena", "sebagai",
    "dalam", "adalah", "atau", "oleh", "agar", "bagi", "para",
    "saat", "setelah", "sebelum", "namun", "tetapi", "hingga",
]
FEATURE_NAMES = [
    "word_count", "sentence_count", "avg_word_length", "avg_sentence_length",
    "sentence_length_variance", "lexical_diversity", "punctuation_density",
    "comma_ratio", "period_ratio", "question_ratio", "exclamation_ratio",
    "semicolon_colon_ratio", "dash_ratio", "digit_char_ratio", "uppercase_ratio",
    "numeric_ratio", "stopword_ratio", "paragraph_count", "avg_paragraph_length",
    "short_word_ratio", "long_word_ratio", "suffix_nya_ratio", "suffix_lah_ratio",
    "suffix_kah_ratio"
] + [f"fw_{w}" for w in FUNCTION_WORDS]

INDONESIAN_STOPWORDS = set([
    "ada", "adanya", "adalah", "adapun", "agak", "agaknya", "agar", "akan", "akankah", "akhir", "akhiri", "akhirnya", "aku", "akulah", "amat", "amatlah", "anda", "andalah", "antar", "antara", "antaranya", "apa", "apaan", "apabila", "apakah", "apalagi", "apatah", "artinya", "asal", "asalkan", "atas", "atau", "ataukah", "ataupun", "awal", "awalnya", "bagai", "bagaikan", "bagaimana", "bagaimanakah", "bagaimanapun", "bagi", "bagian", "bahkan", "bahwa", "bahwasanya", "baik", "baiklah", "bakal", "bakalan", "balik", "banyak", "bapak", "baru", "bawah", "beberapa", "begini", "beginian", "beginikah", "beginilah", "begitu", "begitukah", "begitulah", "begitupun", "bekas", "belakang", "belakangan", "belum", "belumlah", "benar", "benarkah", "benarlah", "berada", "berakhir", "berakhirlah", "berakhirnya", "berapa", "berapakah", "berapalah", "berapapun", "berarti", "berawal", "berbagai", "berdatangan", "beri", "berikan", "berikut", "berikutnya", "berjumlah", "berkali", "berkenaan", "berlainan", "berlalu", "berlangsung", "berlebihan", "bermacam", "bermaksud", "bermula", "bersama", "bersiap", "bertanya", "berturut", "bertutur", "berupa", "besar", "besok", "betul", "betulkah", "biasa", "biasanya", "bila", "bilakah", "bilamana", "bisa", "bisakah", "boleh", "bolehkah", "bolehlah", "buat", "bukan", "bukankah", "bukanlah", "bukannya", "bulan", "bung", "cara", "caranya", "cukup", "cukupkah", "cukuplah", "cuma", "dahulu", "dalam", "dan", "dapat", "dari", "daripada", "datang", "dekat", "demi", "demikian", "demikianlah", "dengan", "depan", "di", "dia", "diakhiri", "diakhirinya", "dialah", "diantara", "diantaranya", "diberi", "diberikan", "diberikannya", "dibuat", "dibuatnya", "didapat", "didatangkan", "digunakan", "diibaratkan", "diibaratkannya", "diingat", "diingatkan", "diinginkan", "dijawab", "dijawabnya", "dijelas", "dijelaskan", "dijelaskannya", "dikarenakan", "dikatakan", "dikatakannya", "dikerjakan", "diketahui", "diketahuinya", "dikira", "dilakukan", "dilalui", "dilihat", "dimaksud", "dimaksudkan", "dimaksudkannya", "dimaksudnya", "diminta", "dimintai", "dimisalkan", "dimulai", "dimulailah", "dimulainya", "dimungkinkan", "dini", "dipastikan", "diperbuat", "diperbuatnya", "dipergunakan", "diperkirakan", "diperlihatkan", "diperlukan", "diperlukannya", "dipersoalkan", "dipertanyakan", "dipunyai", "diri", "dirinya", "disampaikan", "disebut", "disebutkan", "disebutkannya", "disini", "disinilah", "ditambahkan", "ditandaskan", "ditanya", "ditanyai", "ditanyakan", "ditegaskan", "ditujukan", "ditunjuk", "ditunjuki", "ditunjukkan", "ditunjukkannya", "ditunjuknya", "dituturkan", "dituturkannya", "diucapkan", "diucapkannya", "diungkapkan", "dong", "dua", "dulu", "empat", "enggak", "enggaknya", "entah", "entahlah", "guna", "gunakan", "hal", "hampir", "hanya", "hanyalah", "hari", "harus", "haruslah", "harusnya", "hendak", "hendaklah", "hendaknya", "hingga", "ia", "ialah", "ibarat", "ibaratkan", "ibaratnya", "ibu", "ikut", "ingat", "ingin", "inginkah", "inginkan", "ini", "inikah", "inilah", "itu", "itukah", "itulah", "jadi", "jadilah", "jadinya", "jangan", "jangankan", "janganlah", "jauh", "jawab", "jawaban", "jawabnya", "jelas", "jelaskan", "jelaslah", "jelasnya", "jika", "jikalau", "juga", "jumlah", "jumlahnya", "justru", "kala", "kalau", "kalaulah", "kalaupun", "kalian", "kami", "kamilah", "kamu", "kamulah", "kan", "kapan", "kapankah", "kapanpun", "karena", "karenanya", "kasus", "kata", "katakan", "katakanlah", "katanya", "ke", "keadaan", "kebetulan", "kecil", "kedua", "keduanya", "keinginan", "kelamaan", "kelihatan", "kelihatannya", "kelima", "keluar", "kembali", "kemudian", "kemungkinan", "kemungkinannya", "kenapa", "kepada", "kepadanya", "kesampaian", "keseluruhan", "keseluruhannya", "keterlaluan", "ketika", "khususnya", "kini", "kinilah", "kira", "kiranya", "kita", "kitalah", "kok", "kurang", "lagi", "lagian", "lah", "lain", "lainnya", "lalu", "lama", "lamanya", "lanjut", "lanjutnya", "lebih", "lewat", "lima", "luar", "macam", "maka", "makanya", "makin", "malah", "malahan", "mampu", "mampukah", "mana", "manakala", "manalagi", "masa", "masalah", "masalahnya", "masih", "masihkah", "masing", "mau", "maupun", "melainkan", "melakukan", "melalui", "melihat", "melihatnya", "memang", "memastikan", "memberi", "memberikan", "membuat", "memerlukan", "memihak", "meminta", "memintakan", "memisalkan", "memperbuat", "mempergunakan", "memperkirakan", "memperlihatkan", "mempersiapkan", "mempersoalkan", "mempertanyakan", "mempunyai", "memulai", "memungkinkan", "menaiki", "menambahkan", "menandaskan", "menanti", "menantikan", "menanya", "menanyai", "menanyakan", "mendapat", "mendapatkan", "mendatang", "mendatangi", "mendatangkan", "menegaskan", "mengakhiri", "mengapa", "mengatakan", "mengatakannya", "mengenai", "mengerjakan", "mengetahui", "menggunakan", "menghendaki", "mengibaratkan", "mengibaratkannya", "mengingat", "mengingatkan", "menginginkan", "mengira", "mengucapkan", "mengucapkannya", "mengungkapkan", "menjadi", "menjawab", "menjelaskan", "menuju", "menunjuk", "menunjuki", "menunjukkan", "menunjuknya", "menurut", "menuturkan", "menyampaikan", "menyangkut", "menyatakan", "menyebutkan", "menyeluruh", "menyiapkan", "merasa", "mereka", "merekalah", "merupakan", "meski", "meskipun", "minta", "mirip", "misal", "misalkan", "misalnya", "mula", "mulai", "mulailah", "mulanya", "mungkin", "mungkinkah", "nah", "naik", "namun", "nanti", "nantinya", "nyaris", "nyatanya", "oleh", "olehnya", "pada", "padahal", "padanya", "pak", "paling", "panjang", "pantas", "para", "pasti", "pastilah", "penting", "pentingnya", "per", "percuma", "perlu", "perlukah", "perlunya", "pernah", "persoalan", "pertama", "pertanyaan", "pertanyakan", "pihak", "pihaknya", "pukul", "pula", "pun", "punya", "rasa", "rasanya", "rata", "rupanya", "saat", "saatnya", "saja", "sajalah", "saling", "sama", "sambil", "sampai", "sampaikan", "sana", "sangat", "sangatlah", "satu", "saya", "sayalah", "se", "sebab", "sebabnya", "sebagai", "sebagaimana", "sebagainya", "sebagian", "sebaik", "sebaiknya", "sebaliknya", "sebanyak", "sebegini", "sebegitu", "sebelum", "sebelumnya", "sebenarnya", "seberapa", "sebesar", "sebetulnya", "sebisanya", "sebuah", "sebut", "sebutlah", "sebutnya", "secara", "secukupnya", "sedang", "sedangkan", "sedemikian", "sedikit", "sedikitnya", "seenaknya", "segala", "segalanya", "segera", "seharusnya", "sehingga", "seingat", "sejak", "sejauh", "sejenak", "sejumlah", "sekadar", "sekadarnya", "sekali", "sekalian", "sekaligus", "sekalipun", "sekarang", "sekaranglah", "sekecil", "seketika", "sekiranya", "sekitar", "sekitarnya", "sekurang", "sekurangnya", "sela", "selain", "selaku", "selalu", "selama", "selamanya", "selanjutnya", "seluruh", "seluruhnya", "semacam", "semakin", "semampu", "semampunya", "semasa", "semasih", "semata", "sementara", "semisal", "semisalnya", "sempat", "semua", "semuanya", "semula", "sendiri", "sendirian", "sendirinya", "seolah", "seperti", "sepertinya", "seperlunya", "sering", "seringnya", "serta", "serupa", "sesaat", "sesama", "sesampai", "sesegera", "seseorang", "sesuatu", "sesuatunya", "sesudah", "sesudahnya", "setelah", "setempat", "setengah", "seterusnya", "setiap", "setiap", "setiba", "setidak", "setidaknya", "setinggi", "seusai", "sewaktu", "siap", "siapa", "siapakah", "siapapun", "sini", "sinilah", "soal", "soalnya", "suatu", "sudah", "sudahkah", "sudahlah", "supaya", "tadi", "tadinya", "tahu", "tahun", "tak", "tampak", "tampaknya", "tandas", "tandasnya", "tanpa", "tanya", "tanyakan", "tanyanya", "tapi", "tenang", "tengah", "tentang", "tentu", "tentulah", "tentunya", "tepat", "terakhir", "terasa", "terbanyak", "terdahulu", "terdapat", "terdiri", "terhadap", "terhadapnya", "teringat", "terjadi", "terjadilah", "terjadinya", "terkira", "terlalu", "terlebih", "terlihat", "termasuk", "ternyata", "tersampaikan", "tersebut", "tersebutlah", "tertentu", "tertuju", "terus", "terutama", "tetap", "tetapi", "tiada", "tiadakah", "tiadalah", "tidak", "tidakkah", "tidaklah", "tiga", "tinggi", "toh", "tunjuk", "turut", "tutur", "tuturnya", "ucap", "ucapnya", "ujar", "ujarnya", "umum", "umumnya", "ungkap", "ungkapnya", "untuk", "usah", "usai", "waduh", "wah", "wahai", "waktu", "waktunya", "walau", "walaupun", "wong", "yaitu", "yakin", "yakni", "yang"
])

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def tokenize_words(text):
    return [t.lower() for t in WORD_RE.findall(normalize_text(text))]

def split_sentences(text):
    sentences = [s.strip() for s in SENTENCE_RE.findall(normalize_text(text))]
    return [s for s in sentences if s]

def safe_div(n, d):
    return n / d if d else 0.0

def extract_stylometry(text):
    raw = str(text)
    words = tokenize_words(raw)
    sentences = split_sentences(raw)
    sentence_lengths = [len(tokenize_words(s)) for s in sentences]
    paragraphs = [p.strip() for p in raw.splitlines() if p.strip()]
    chars = [c for c in raw if not c.isspace()]
    punctuation = [c for c in raw if c in string.punctuation]
    uppercase_chars = [c for c in raw if c.isupper()]
    numeric_tokens = [w for w in words if w.isdigit()]
    stop_count = [w for w in words if w in INDONESIAN_STOPWORDS]

    wc = len(words)
    sc = len(sentences)
    cc = len(chars)
    avg_sent_len = safe_div(wc, sc)
    sent_var = safe_div(sum((x - avg_sent_len) ** 2 for x in sentence_lengths), len(sentence_lengths))

    feats = [
        float(wc),
        float(sc),
        safe_div(sum(len(w) for w in words), wc),
        avg_sent_len,
        sent_var,
        safe_div(len(set(words)), wc),
        safe_div(len(punctuation), cc),
        safe_div(raw.count(","), cc),
        safe_div(raw.count("."), cc),
        safe_div(raw.count("?"), cc),
        safe_div(raw.count("!"), cc),
        safe_div(raw.count(";") + raw.count(":"), cc),
        safe_div(raw.count("-"), cc),
        safe_div(sum(ch.isdigit() for ch in raw), cc),
        safe_div(len(uppercase_chars), cc),
        safe_div(len(numeric_tokens), wc),
        safe_div(len(stop_count), wc),
        float(len(paragraphs) or 1),
        safe_div(sum(len(tokenize_words(p)) for p in paragraphs), len(paragraphs) or 1),
        safe_div(sum(1 for w in words if len(w) <= 3), wc),
        safe_div(sum(1 for w in words if len(w) >= 8), wc),
        safe_div(sum(1 for w in words if w.endswith("nya")), wc),
        safe_div(sum(1 for w in words if w.endswith("lah")), wc),
        safe_div(sum(1 for w in words if w.endswith("kah")), wc),
    ]

    for fw in FUNCTION_WORDS:
        feats.append(safe_div(words.count(fw), wc))

    return feats

print("Extracting stylometric features...")
final_df["stylometry"] = final_df["text"].apply(extract_stylometry)
sty_matrix = np.array(final_df["stylometry"].tolist(), dtype=np.float32)

# =========================================================
# 3. TRAIN TEST SPLITS
# =========================================================
X_text = final_df["text"].values
X_sty = sty_matrix
y = final_df["label"].values
author_names = final_df["author"].values
article_ids = np.arange(len(final_df))

X_train_text, X_test_text, X_train_sty, X_test_sty, y_train, y_test, author_train, author_test, id_train, id_test = train_test_split(
    X_text, X_sty, y, author_names, article_ids,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

X_train_text, X_val_text, X_train_sty, X_val_sty, y_train, y_val, author_train, author_val, id_train, id_val = train_test_split(
    X_train_text, X_train_sty, y_train, author_train, id_train,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train,
)

scaler = StandardScaler()
X_train_sty = scaler.fit_transform(X_train_sty).astype(np.float32)
X_val_sty = scaler.transform(X_val_sty).astype(np.float32)
X_test_sty = scaler.transform(X_test_sty).astype(np.float32)

# =========================================================
# 4. CHUNKING & DATA EXPANSION
# =========================================================
def chunk_text(text, n_chunks=2):
    words = str(text).split()
    if len(words) <= 1 or n_chunks <= 1:
        return [str(text).strip()]

    boundaries = np.linspace(0, len(words), n_chunks + 1, dtype=int)
    pieces = []
    for i in range(n_chunks):
        piece = " ".join(words[boundaries[i]:boundaries[i + 1]]).strip()
        if piece:
            pieces.append(piece)
    return pieces if pieces else [str(text).strip()]

def expand_split(texts, stylometry, labels, authors, ids, chunk_human=True):
    out_texts, out_sty, out_labels, out_authors, out_ids = [], [], [], [], []
    for text, sty, label, author, aid in zip(texts, stylometry, labels, authors, ids):
        should_chunk = chunk_human and (author != "AI")
        pieces = chunk_text(text, CHUNKS_PER_HUMAN_TEXT) if should_chunk else [str(text).strip()]
        for piece in pieces:
            out_texts.append(piece)
            out_sty.append(sty)
            out_labels.append(int(label))
            out_authors.append(author)
            out_ids.append(int(aid))
    return (
        out_texts,
        np.asarray(out_sty, dtype=np.float32),
        np.asarray(out_labels, dtype=np.int64),
        np.asarray(out_authors),
        np.asarray(out_ids, dtype=np.int64),
    )

print("Expanding and chunking datasets...")
X_train_text_c, X_train_sty_c, y_train_c, auth_train_c, id_train_c = expand_split(
    X_train_text, X_train_sty, y_train, author_train, id_train, chunk_human=True
)
X_val_text_c, X_val_sty_c, y_val_c, auth_val_c, id_val_c = expand_split(
    X_val_text, X_val_sty, y_val, author_val, id_val, chunk_human=True
)
X_test_text_c, X_test_sty_c, y_test_c, auth_test_c, id_test_c = expand_split(
    X_test_text, X_test_sty, y_test, author_test, id_test, chunk_human=True
)

# =========================================================
# 5. DATASET & DATALOADERS
# =========================================================
class AuthorshipDataset(Dataset):
    def __init__(self, texts, stylometry, labels, article_ids, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.stylometry = stylometry
        self.labels = labels
        self.article_ids = article_ids
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "stylometry": torch.tensor(self.stylometry[idx], dtype=torch.float),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "article_id": torch.tensor(self.article_ids[idx], dtype=torch.long),
        }

def make_weighted_sampler(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    counts = np.maximum(counts, 1)
    weights = np.array([1.0 / counts[label] for label in labels], dtype=np.float64)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)

def make_loaders(tokenizer):
    train_ds = AuthorshipDataset(X_train_text_c, X_train_sty_c, y_train_c, id_train_c, tokenizer)
    val_ds = AuthorshipDataset(X_val_text_c, X_val_sty_c, y_val_c, id_val_c, tokenizer)
    test_ds = AuthorshipDataset(X_test_text_c, X_test_sty_c, y_test_c, id_test_c, tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=16 if torch.cuda.device_count() >= 2 else 8,
        sampler=make_weighted_sampler(y_train_c),
        num_workers=2 if os.name != "nt" else 0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=16 if torch.cuda.device_count() >= 2 else 8,
        shuffle=False,
        num_workers=2 if os.name != "nt" else 0,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=16 if torch.cuda.device_count() >= 2 else 8,
        shuffle=False,
        num_workers=2 if os.name != "nt" else 0,
        pin_memory=torch.cuda.is_available(),
    )
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

# =========================================================
# 6. MODEL SPECIFICATION (Aligned Architecture)
# =========================================================
class DualChannelModel(nn.Module):
    def __init__(self, model_name, num_classes, num_sty_features):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size

        self.sty_fc1 = nn.Linear(num_sty_features, 64)
        self.sty_norm1 = nn.LayerNorm(64)
        self.sty_fc2 = nn.Linear(64, 64)
        self.sty_norm2 = nn.LayerNorm(64)
        self.sty_relu = nn.ReLU()
        self.sty_dropout = nn.Dropout(0.30)

        self.classifier = nn.Sequential(
            nn.Linear(hidden + 64, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

    def forward_stylometry(self, stylometry: torch.Tensor) -> torch.Tensor:
        s1 = self.sty_dropout(self.sty_relu(self.sty_norm1(self.sty_fc1(stylometry))))
        s2 = self.sty_dropout(self.sty_relu(self.sty_norm2(self.sty_fc2(s1))))
        return s1 + s2

    def forward(self, input_ids, attention_mask, stylometry, return_attentions=False):
        if return_attentions:
            self.backbone.config.output_attentions = True
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_attentions=return_attentions)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        sty_vec = self.forward_stylometry(stylometry)
        fused = torch.cat([cls_vec, sty_vec], dim=1)
        logits = self.classifier(fused)
        
        if return_attentions:
            return logits, outputs.attentions
        return logits

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

# =========================================================
# 7. EVALUATION PIPELINE
# =========================================================
@torch.no_grad()
def predict_grouped(model, loader, ai_label_idx=None):
    model.eval()
    logits_by_group = defaultdict(list)
    label_by_group = {}

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        stylometry = batch["stylometry"].to(device)
        labels_batch = batch["label"].cpu().numpy()
        group_ids = batch["article_id"].cpu().numpy()

        outputs = model(input_ids, attention_mask, stylometry)
        outputs = outputs.detach().cpu().numpy()

        for i, gid in enumerate(group_ids):
            logits_by_group[int(gid)].append(outputs[i])
            label_by_group[int(gid)] = int(labels_batch[i])

    y_true = []
    y_pred = []
    for gid in sorted(logits_by_group.keys()):
        mean_logits = np.mean(logits_by_group[gid], axis=0)
        pred = int(np.argmax(mean_logits))
        y_pred.append(pred)
        y_true.append(label_by_group[gid])

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    ai_f1 = 0.0
    if ai_label_idx is not None:
        if ai_label_idx in y_true or ai_label_idx in y_pred:
            ai_f1 = f1_score(y_true, y_pred, labels=[ai_label_idx], average=None, zero_division=0)[0]
    
    return acc, prec, f1, ai_f1, y_true, y_pred

# =========================================================
# 8. TRAINING MODULE
# =========================================================
def train_model(model, train_loader, val_loader):
    ai_label_idx = label2id.get("AI", None)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    base_model = unwrap_model(model)
    backbone_params = list(base_model.backbone.parameters())
    head_params = [p for n, p in base_model.named_parameters() if not n.startswith('backbone.')]
    
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": LR_BASE},
            {"params": head_params, "lr": LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = max(len(train_loader) * EPOCHS, 1)
    warmup_steps = max(int(0.10 * total_steps), 1)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_f1 = -1.0
    best_state = None
    patience_counter = 0

    print(f"\nStarting {DISPLAY_NAME} Training Loop...")
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False, disable=not SHOW_TQDM, mininterval=5.0)
        for batch in iterator:
            optimizer.zero_grad(set_to_none=True)

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            stylometry = batch["stylometry"].to(device)
            labels_batch = batch["label"].to(device)

            outputs = model(input_ids, attention_mask, stylometry)
            loss = criterion(outputs, labels_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()

        train_loss /= len(train_loader)
        
        _, _, train_f1, train_ai_f1, _, _ = predict_grouped(model, train_loader, ai_label_idx)
        _, _, val_f1, val_ai_f1, _, _ = predict_grouped(model, val_loader, ai_label_idx)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val AI F1: {val_ai_f1:.4f}")
        
        if train_f1 - val_f1 > 0.15:
            print(f"  --> [Overfitting Warn] Train F1 is {train_f1 - val_f1:.4f} higher than Val F1.")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered at epoch {epoch + 1}!")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# =========================================================
# 9. RUN MODEL TRAINING
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = make_loaders(tokenizer)

model = DualChannelModel(MODEL_NAME, NUM_CLASSES, X_train_sty_c.shape[1])

model = model.float().to(device)
if torch.cuda.device_count() >= 2:
    model = nn.DataParallel(model)

model = train_model(model, train_loader, val_loader)

ai_label_idx = label2id.get("AI", None)
acc, prec, f1, ai_f1, y_true, y_pred = predict_grouped(model, test_loader, ai_label_idx=ai_label_idx)

print("\n" + "=" * 50)
print(f"             FINAL RESULTS: {DISPLAY_NAME}")
print("=" * 50)
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"AI F1     : {ai_f1:.4f}")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

# Save test set confusion matrix plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title(f"Test Set Confusion Matrix - {DISPLAY_NAME}")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plot_prefix = DISPLAY_NAME.lower().replace(" ", "_").replace("-", "_")
plt.savefig(f"{plot_prefix}_confusion_matrix.png", dpi=150)
plt.close()
print(f"Saved test confusion matrix plot to '{plot_prefix}_confusion_matrix.png'.")

# =========================================================
# 10. EXPLAINABLE AI (xAI) - 5 RANDOM TEST SAMPLE PLOTS
# =========================================================
def run_xai_explainability_suite(model, tokenizer, num_samples=5):
    print(f"\nGenerating full xAI Explainability plots for {num_samples} random test samples...")
    base_model = unwrap_model(model)
    base_model.eval()

    model_id = base_model.backbone.config._name_or_path
    eager_backbone = AutoModel.from_pretrained(model_id, attn_implementation="eager").float()
    eager_backbone.load_state_dict(base_model.backbone.state_dict())
    eager_backbone.to(device)
    original_backbone = base_model.backbone
    base_model.backbone = eager_backbone

    random.seed(SEED)
    sample_indices = random.sample(range(len(X_test_text_c)), num_samples)
    plot_prefix = DISPLAY_NAME.lower().replace(" ", "_").replace("-", "_")

    for idx_num, sample_idx in enumerate(sample_indices):
        text = X_test_text_c[sample_idx]
        sty_features = X_test_sty_c[sample_idx]
        true_author = auth_test_c[sample_idx]

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        sty_tensor = torch.tensor([sty_features], dtype=torch.float, device=device, requires_grad=True)

        logits, attentions = base_model(input_ids, attention_mask, sty_tensor, return_attentions=True)
        pred_idx = torch.argmax(logits, dim=1).item()
        pred_author = id2label[pred_idx]
        confidence = float(torch.softmax(logits, dim=1)[0, pred_idx])

        print(f"  --> Sample {idx_num + 1}: True={true_author}, Pred={pred_author} ({confidence:.3f})")

        # 1. Token Attention Heatmap (Seaborn Heatmap of CLS/<s> attention to first 30 tokens)
        last_layer_attention = attentions[-1][0]
        avg_attention = torch.mean(last_layer_attention, dim=0)
        cls_attention = avg_attention[0].detach().cpu().numpy()
        
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
        valid_indices = [i for i, tok in enumerate(tokens) if tok not in ["[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"]][:30]
        
        if valid_indices:
            sample_tokens = [tokens[i] for i in valid_indices]
            sample_atts = cls_attention[valid_indices]
            
            if sample_atts.max() > sample_atts.min():
                sample_atts = (sample_atts - sample_atts.min()) / (sample_atts.max() - sample_atts.min())
            
            plt.figure(figsize=(15, 2))
            sns.heatmap([sample_atts], annot=True, fmt=".2f", cmap="YlOrRd", 
                        xticklabels=sample_tokens, yticklabels=False, cbar=False)
            plt.title(f"Sample {idx_num + 1} Token Attention Weights (Normalized) | True: {true_author} | Pred: {pred_author}")
            plt.tight_layout()
            plt.savefig(f"{plot_prefix}_sample_{idx_num + 1}_attention_heatmap.png", dpi=150)
            plt.close()

            # 2. Top-15 Attention Tokens (Bar Chart)
            token_att_dict = defaultdict(float)
            for tok, att in zip(tokens, cls_attention):
                if tok not in ["[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"]:
                    clean_tok = tok.replace("##", "").replace(" ", "")
                    token_att_dict[clean_tok] += float(att)
                    
            token_att_pairs = list(token_att_dict.items())
            token_att_pairs.sort(key=lambda x: x[1], reverse=True)
            top_15_tokens = token_att_pairs[:15]
            
            plt.figure(figsize=(8, 5))
            tok_labels = [x[0] for x in top_15_tokens]
            tok_values = [x[1] for x in top_15_tokens]
            sns.barplot(x=tok_values, y=tok_labels, palette="viridis")
            plt.title(f"Sample {idx_num + 1} Top 15 Attention Weights ({DISPLAY_NAME})")
            plt.xlabel("Attention Score")
            plt.tight_layout()
            plt.savefig(f"{plot_prefix}_sample_{idx_num + 1}_top_tokens.png", dpi=150)
            plt.close()

        # 3. Stylometric Attribution via Autograd
        pred_logit = logits[0, pred_idx]
        base_model.zero_grad()
        pred_logit.backward()

        sty_grad = sty_tensor.grad.squeeze(0).cpu().numpy()
        attributions = sty_grad * sty_features

        sty_att_pairs = [
            {"feature": name, "importance": float(attr)}
            for name, attr in zip(FEATURE_NAMES, attributions)
        ]
        sty_att_pairs.sort(key=lambda x: abs(x["importance"]), reverse=True)
        top_10_sty = sty_att_pairs[:10]

        plt.figure(figsize=(9, 5))
        feat_labels = [x["feature"] for x in top_10_sty]
        feat_values = [x["importance"] for x in top_10_sty]
        colors = ["green" if val >= 0 else "red" for val in feat_values]
        
        sns.barplot(x=feat_values, y=feat_labels, palette=colors)
        plt.title(f"Sample {idx_num + 1} Top 10 Stylometric Attributions ({DISPLAY_NAME})\n(Green = Positive/Supporting, Red = Negative/Opposing)")
        plt.xlabel("Attribution Score (Gradient * Feature)")
        plt.axvline(x=0, color="gray", linestyle="--")
        plt.tight_layout()
        plt.savefig(f"{plot_prefix}_sample_{idx_num + 1}_stylometry_drivers.png", dpi=150)
        plt.close()

    base_model.backbone = original_backbone
    print(f"\n✅ Explainable AI suite finished successfully! Generated 3 files per sample with prefix '{plot_prefix}'.")

run_xai_explainability_suite(model, tokenizer, num_samples=5)

# =========================================================
# 11. EXPORT MODEL ARTIFACTS
# =========================================================
print("\nExporting model artifacts...")
EXPORT_DIR.mkdir(exist_ok=True)

state = unwrap_model(model).state_dict()
torch.save(state, EXPORT_DIR / "feature_fusion_transformer.pt")
print(f"  --> Saved FeatureFusionTransformer weights to: {EXPORT_DIR / 'feature_fusion_transformer.pt'}")

tokenizer.save_pretrained(str(EXPORT_DIR / "tokenizer"))
print(f"  --> Saved tokenizer to: {EXPORT_DIR / 'tokenizer/'}")

joblib.dump(scaler, EXPORT_DIR / "standard_scaler.joblib")
print(f"  --> Saved StandardScaler to: {EXPORT_DIR / 'standard_scaler.joblib'}")

with open(EXPORT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, ensure_ascii=False, indent=2)
print(f"  --> Saved label map to: {EXPORT_DIR / 'label_map.json'}")

print(f"\n✅ All artifacts exported to: {EXPORT_DIR.resolve()}")
print(f"Finished {DISPLAY_NAME} training script.")

Using device: cuda
Loading merged dataset from: /kaggle/input/datasets/xandertrevor/stylometry/merged_dataset.csv
Selected top 10 human authors based on article count:
  01. Muchamad Aly Reza (112 articles)
  02. Habib Allbi Ferdian (96 articles)
  03. Ahmad Effendi (95 articles)
  04. Mercy Raya (90 articles)
  05. Aisyah Amira Wakang (90 articles)
  06. Gitario Vista Inasis (85 articles)
  07. Febryantino Nur Pratama (67 articles)
  08. Azalia Amadea (65 articles)
  09. Shofiatunnisa Azizah (59 articles)
  10. Ela Nurlaela (58 articles)
Unified dataset contains 1021 rows across 11 classes (AI + 10 humans).
Extracting stylometric features...
Expanding and chunking datasets...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

...

Epoch 05 | Train Loss: 1.1052 | Train F1: 0.7469 | Val F1: 0.6552 | Val AI F1: 0.8519


Epoch 6/6:   0%|          | 0/78 [00:00<?, ?it/s]

Epoch 06 | Train Loss: 0.9843 | Train F1: 0.7815 | Val F1: 0.6246 | Val AI F1: 0.8421
  --> [Overfitting Warn] Train F1 is 0.1570 higher than Val F1.

             FINAL RESULTS: Proposed mDeBERTa-v3 + Stylometry
Accuracy  : 0.6878
Precision : 0.7036
F1 Score  : 0.6581
AI F1     : 0.8750

Classification Report:
                         precision    recall  f1-score   support

                     AI       0.90      0.85      0.88        41
          Ahmad Effendi       0.71      0.26      0.38        19
    Aisyah Amira Wakang       0.27      0.50      0.35        18
          Azalia Amadea       0.60      0.69      0.64        13
           Ela Nurlaela       0.71      0.83      0.77        12
Febryantino Nur Pratama       0.87      1.00      0.93        13
   Gitario Vista Inasis       0.65      0.88      0.75        17
    Habib Allbi Ferdian       0.68      0.89      0.77        19
             Mercy Raya       0.84      0.89      0.86        18
      Muchamad Aly Reza       0.90  

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

  --> Sample 1: True=Habib Allbi Ferdian, Pred=Habib Allbi Ferdian (0.507)
  --> Sample 2: True=Ela Nurlaela, Pred=Ela Nurlaela (0.392)
  --> Sample 3: True=Azalia Amadea, Pred=Ela Nurlaela (0.365)
  --> Sample 4: True=Mercy Raya, Pred=Mercy Raya (0.772)
  --> Sample 5: True=Febryantino Nur Pratama, Pred=Febryantino Nur Pratama (0.701)

✅ Explainable AI suite finished successfully! Generated 3 files per sample with prefix 'proposed_mdeberta_v3_+_stylometry'.

Exporting model artifacts...
  --> Saved FeatureFusionTransformer weights to: model_artifacts/feature_fusion_transformer.pt
  --> Saved tokenizer to: model_artifacts/tokenizer
  --> Saved StandardScaler to: model_artifacts/standard_scaler.joblib
  --> Saved label map to: model_artifacts/label_map.json

✅ All artifacts exported to: /kaggle/working/model_artifacts
Finished Proposed mDeBERTa-v3 + Stylometry training script.
